# 01 — Valley bottom (VBET) for an MGRS tile

Delineates the riparian valley bottom that every later notebook clips to. Runs the
**three-tool chain** — breach → HAND → slope — over one MGRS 100-km square.

**Reads** NHDPlus flowlines (drainage area), 3DEP staged DEM tiles
**Writes** `<run>_valley_bottom.gpkg`, `<run>_valley_mask_30m.tif`, a manifest in `runs/`
**Status** Phase 1b — rewritten, **not yet run on 13TFJ**

> **Why there is no flow accumulation here.** The retired chain used
> `D8Pointer → D8FlowAccumulation → ExtractStreams`. Flow accumulation is the only
> globally-dependent step: it needs the whole upstream watershed, so inside a clipped
> tile the streams entering from outside start at zero and HAND comes out wrong. The NHD
> flowlines are the stream network instead — they already carry surveyed `totdasqkm`.
> Every remaining step is local, so **tile seams stop mattering.**

> `01a_DEM_Prefetch` is folded into this notebook (sections 4). The WhiteboxTools binary
> is downloaded and persisted by `scripts/setup_cyverse.sh`, not here.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json, math, time, subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import rasterio.plot
from rasterio.features import shapes, rasterize
from shapely.geometry import box, shape
from shapely.ops import unary_union
from scipy.ndimage import label, binary_fill_holes
from osgeo import gdal
import whitebox


def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

print("Imports OK")
print(f"  repo : {REPO}")
print(f"  data : {DATA_DIR}")

## 1. Configuration

Every parameter lives here. Pointing this notebook at another square is a one-line edit —
`TILE` drives the extent, the CRS, and the DEM tile list.

In [ ]:
# ---- Which tile ----
# 13TFJ is the pilot square: it holds Angostura, Buffalo Gap, Red Shirt and Scenic.
# The corridor spans nine squares across UTM zones 13 AND 14, so the CRS is derived
# from the tile rather than hardcoded -- EPSG:32613 is not right corridor-wide.
TILE        = "13TFJ"
TILE_BUFFER_M = 5_000          # pad so the valley bottom is not clipped at the seam

# ---- Grid ----
# Change DEM_RES_M alone. Everything resolution-dependent below is derived from it,
# because the failure mode of NOT deriving them is silent: the run still completes.
DEM_RES_M   = 30                                # 30 = 3DEP 1 arc-sec, 10 = 1/3 arc-sec
DEM_PRODUCT = {30: "1", 10: "13"}[DEM_RES_M]    # KeyError beats upsampling 30 m data to 10 m

# ---- Hydrology ----
# Least-cost breaching carves through blockages up to this far. Set it in METRES: a fixed
# cell count quietly changes the PHYSICAL search distance when the grid changes — 100 cells
# is 3 km at 30 m but only 1 km at 10 m, which is a different hydrological assumption.
BREACH_DIST_M     = 3_000
BREACH_DIST_CELLS = max(1, round(BREACH_DIST_M / DEM_RES_M))

# ---- VBET thresholds by drainage area class ----
# Each: drainage area window (km2), max height above stream (m), max slope (deg), buffer (m)
#
# WARNING — `slope_deg` is NOT resolution-portable. Slope measured on a finer DEM is
# systematically steeper, because a smaller cell resolves local relief that a coarse cell
# averages away. These values were tuned at 30 m; reused unchanged at 10 m they exclude more
# land and the valley bottom shrinks for reasons of grid size, not geomorphology. Section 9
# measures that shift so the thresholds can be re-tuned on evidence rather than guessed.
VBET_CLASSES = [
    {"label": "large",  "da_min": 1000, "da_max": 1e9,  "hand_m": 12, "slope_deg": 6,  "buffer_m": 1000},
    {"label": "medium", "da_min": 100,  "da_max": 1000, "hand_m": 8,  "slope_deg": 8,  "buffer_m": 500},
    {"label": "small",  "da_min": 0,    "da_max": 100,  "hand_m": 4,  "slope_deg": 12, "buffer_m": 200},
]
MIN_PATCH_HA = 1.0             # drop valley-bottom patches smaller than this

# Which size classes go into the analysis mask. Small-class reaches are ephemeral
# badlands draws -- lots of area, not cottonwood gallery habitat. Every reach is still
# classed and written out, so adding "small" back is a one-word change, not a re-run.
ANALYSIS_CLASSES = ["large", "medium"]

# ---- Outputs ----
RUN_NAME    = f"vbet_{TILE}_{DEM_RES_M}m"
FIG_SUBDIR = RUN_NAME               # figures/<run>/
OUT_DIR     = DATA_DIR / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_GPKG = OUT_DIR / f"{RUN_NAME}_valley_bottom.gpkg"

## 2. The tile

The MGRS grid is deterministic, so the square's bounds come from arithmetic rather than a
lookup table or an extra package. HLS and Sentinel-2 granules already ship on this grid,
which is why it is the tiling unit.

In [ ]:
COL_SETS = ["ABCDEFGH", "JKLMNPQR", "STUVWXYZ"]
ROW_SETS = ["ABCDEFGHJKLMNPQRSTUV", "FGHJKLMNPQRSTUVABCDE"]
BANDS    = "CDEFGHJKLMNPQRSTUVWX"        # latitude bands, 8 deg each, starting at 80S

def mgrs_square(tile):
    """Projected bounds (minx, miny, maxx, maxy) and EPSG code of an MGRS 100-km square.

    Column letters repeat every 3 zones and map to eastings 100-800 km; row letters
    repeat every 2 zones and every 2,000,000 m of northing.
    """
    zone, band, col, row = int(tile[:2]), tile[2], tile[3], tile[4]
    epsg = (32600 if band >= "N" else 32700) + zone

    minx = (COL_SETS[(zone - 1) % 3].index(col) + 1) * 100_000
    r    = ROW_SETS[(zone - 1) % 2].index(row) * 100_000

    # Northing repeats every 2,000,000 m -- pick the cycle this latitude band sits in.
    lat0   = -80 + 8 * BANDS.index(band)
    approx = (lat0 + 4) * 110_574
    if band < "N":
        approx += 10_000_000
    miny = r + round((approx - r) / 2_000_000) * 2_000_000
    return (minx, miny, minx + 100_000, miny + 100_000), epsg


(TX0, TY0, TX1, TY1), EPSG = mgrs_square(TILE)
CRS_PROJ = f"EPSG:{EPSG}"

tile_proj = gpd.GeoDataFrame(geometry=[box(TX0, TY0, TX1, TY1)], crs=CRS_PROJ)
work_proj = gpd.GeoDataFrame(
    geometry=[box(TX0 - TILE_BUFFER_M, TY0 - TILE_BUFFER_M,
                  TX1 + TILE_BUFFER_M, TY1 + TILE_BUFFER_M)], crs=CRS_PROJ)
work_wgs  = work_proj.to_crs(4326)
WORK_BOUNDS_WGS = tuple(work_wgs.total_bounds)

side = (100_000 + 2 * TILE_BUFFER_M) / DEM_RES_M
print(f"Tile    : {TILE} -> {CRS_PROJ}")
print(f"Bounds  : {TX0:,} - {TX1:,} E, {TY0:,} - {TY1:,} N (+{TILE_BUFFER_M/1000:.0f} km buffer)")
print(f"Grid    : {side:,.0f} x {side:,.0f} cells ({side**2/1e6:.1f} M) at {DEM_RES_M} m")

## 3. Flowlines with drainage area

`totdasqkm` is what assigns each reach its VBET size class. Without it every reach
silently falls through to `medium` and the Cheyenne main stem gets headwater thresholds —
the bug that made the old chain look plausible while being wrong.

In [ ]:
FLOW_CACHE = OUT_DIR / f"{RUN_NAME}_flowlines.gpkg"

if FLOW_CACHE.exists():
    flw = gpd.read_file(FLOW_CACHE, layer="flowlines").to_crs(CRS_PROJ)
    FLOWLINE_SOURCE = FLOW_CACHE.name
    print(f"Flowlines from cache: {len(flw):,} reaches")
else:
    from pynhd import WaterData
    # nhdflowline_network already carries totdasqkm -- no separate VAA join needed.
    flw = WaterData("nhdflowline_network").bybox(WORK_BOUNDS_WGS).to_crs(CRS_PROJ)
    flw = gpd.clip(flw, work_proj)
    flw = flw.rename(columns=str.lower).set_geometry("geometry")
    flw.to_file(FLOW_CACHE, layer="flowlines", driver="GPKG")
    FLOWLINE_SOURCE = "NHDPlus v2 nhdflowline_network (WaterData bybox)"
    print(f"Flowlines from NHDPlus v2: {len(flw):,} reaches")

assert "totdasqkm" in flw.columns, f"No drainage area column. Got: {list(flw.columns)}"
print(f"  drainage area : {flw.totdasqkm.min():.1f} - {flw.totdasqkm.max():,.0f} km2")
print(f"  channel length: {flw.geometry.length.sum() / 1000:,.1f} km")

In [ ]:
# ---- Assign each reach to a VBET size class ----
da = pd.to_numeric(flw["totdasqkm"], errors="coerce")
flw["vbet_class"] = np.select(
    [(da >= c["da_min"]) & (da < c["da_max"]) for c in VBET_CLASSES],
    [c["label"] for c in VBET_CLASSES],
    default="small",
)

print("Reaches per class:")
for c in VBET_CLASSES:
    sel = flw[flw.vbet_class == c["label"]]
    print(f"  {c['label']:6s} (HAND<{c['hand_m']:2d}m, slope<{c['slope_deg']:2d}deg, "
          f"buf {c['buffer_m']:4d}m): {len(sel):5,} reaches, "
          f"{sel.geometry.length.sum() / 1000:8,.1f} km")

## 4. DEM

Static staged 3DEP COG tiles from the public USGS S3 bucket. **Not `py3dep.get_dem()`** —
that hits the 3DEP *dynamic* service which renders elevation on demand, fine for a few
hundred km² and effectively unusable at tile scale. Downloads are resumable and cached.

In [ ]:
TILE_DIR = DATA_DIR / "dem_tiles"
TILE_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = ("https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/"
            "{p}/TIFF/current/{t}/USGS_{p}_{t}.tif")

def tiles_for_bbox(minx, miny, maxx, maxy):
    """1-degree 3DEP tile names covering a WGS84 bbox.

    Tiles are named by their NORTHWEST corner: n44w104 spans lat [43, 44], lon [-104, -103].
    """
    lats = range(math.floor(miny) + 1, math.ceil(maxy) + 1)
    lons = range(math.floor(-maxx) + 1, math.ceil(-minx) + 1)
    return [f"n{a:02d}w{o:03d}" for a in lats for o in lons]

TILES = tiles_for_bbox(*WORK_BOUNDS_WGS)
tile_paths, TILE_URLS = [], []

for t in TILES:
    url  = BASE_URL.format(p=DEM_PRODUCT, t=t)
    dest = TILE_DIR / f"USGS_{DEM_PRODUCT}_{t}.tif"
    TILE_URLS.append(url)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  {dest.name}: cached ({dest.stat().st_size / 1e6:.0f} MB)")
    else:
        print(f"  {dest.name}: downloading…")
        r = subprocess.run(["curl", "-fL", "-C", "-", "--retry", "3", "-o", str(dest), url],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"Download failed for {t}: {r.stderr[-300:]}")
        print(f"    done ({dest.stat().st_size / 1e6:.0f} MB)")
    tile_paths.append(str(dest))

print(f"\n{len(tile_paths)} tile(s) covering {TILE}")

In [ ]:
# Mosaic -> reproject -> clip, in one GDAL pass.
DEM_PATH = OUT_DIR / f"{RUN_NAME}_dem_{DEM_RES_M}m.tif"
CUTLINE  = OUT_DIR / "_tile_cutline.gpkg"
work_wgs[["geometry"]].to_file(CUTLINE, layer="cutline", driver="GPKG")

if DEM_PATH.exists():
    print(f"{DEM_PATH.name} exists — delete it to rebuild.")
else:
    vrt = str(OUT_DIR / "_dem_tiles.vrt")
    gdal.BuildVRT(vrt, tile_paths)
    gdal.Warp(
        str(DEM_PATH), vrt,
        dstSRS=CRS_PROJ, xRes=DEM_RES_M, yRes=DEM_RES_M, resampleAlg="bilinear",
        cutlineDSName=str(CUTLINE), cropToCutline=True,
        dstNodata=-9999.0, outputType=gdal.GDT_Float32,
        creationOptions=["COMPRESS=DEFLATE", "TILED=YES", "NUM_THREADS=ALL_CPUS"],
    )
    print(f"  wrote {DEM_PATH.name} ({DEM_PATH.stat().st_size / 1e6:.1f} MB)")

with rasterio.open(DEM_PATH) as src:
    dem_shape, dem_transform, dem_profile = src.shape, src.transform, src.profile.copy()
    dem = src.read(1, masked=True)
    ext = rasterio.plot.plotting_extent(src)

print(f"  {dem_shape[1]:,} x {dem_shape[0]:,} cells | "
      f"elevation {dem.min():.0f} - {dem.max():.0f} m")

## 5. WhiteboxTools

The binary is downloaded and persisted by `scripts/setup_cyverse.sh`, which also bakes
`WBT_PATH` into the kernel spec. This cell only *uses* what is already there.

In [ ]:
# WBT_PATH must be set BEFORE constructing WhiteboxTools: the constructor calls
# download_wbt(), which returns early when that variable is set. Without it, CyVerse
# re-downloads the ~200 MB binary every session because /opt/conda does not persist.
WBT_PERSIST_DIR = Path(os.environ.get("WBT_PATH", Path.home() / "data-store" / "bin" / "WBT"))
WBT_EXE = "whitebox_tools.exe" if sys.platform.startswith("win") else "whitebox_tools"
have_persisted = (WBT_PERSIST_DIR / WBT_EXE).exists()

if have_persisted:
    os.environ["WBT_PATH"] = str(WBT_PERSIST_DIR)
    print(f"Using persistent WhiteboxTools at {WBT_PERSIST_DIR}")
else:
    print("No persisted binary — WhiteboxTools downloads ~200 MB on first use.\n"
          "  On CyVerse run `bash scripts/setup_cyverse.sh` instead of paying this each session.")

wbt = whitebox.WhiteboxTools()
wbt.verbose = False
if have_persisted:
    wbt.set_whitebox_dir(str(WBT_PERSIST_DIR))
wbt.set_max_procs(-1)                        # -1 = all cores
wbt.set_working_dir(str(OUT_DIR.resolve()))  # WBT resolves bare filenames against this

WBT_VERSION = wbt.version().splitlines()[0].strip()
print(WBT_VERSION)

## 6. Hydrology — three tools

```
BreachDepressionsLeastCost  ->  ElevationAboveStream (HAND)  ->  Slope
```

All three are in the **MIT core** binary, not the bundled plugins. `elevation_above_stream`
takes no flow pointer — it derives its own flow paths — which is why `D8Pointer` left with
`D8FlowAccumulation`.

In [ ]:
# ---- Rasterize the NHD flowlines as the stream network ----
# Background must be 0, not nodata, or HAND will not propagate away from the channel.
streams_path = OUT_DIR / f"{RUN_NAME}_streams_{DEM_RES_M}m.tif"

stream_arr = rasterize(
    ((geom, 1) for geom in flw.geometry),
    out_shape=dem_shape, transform=dem_transform,
    fill=0, dtype="float32", all_touched=True,
)

prof = dem_profile.copy()
prof.update(dtype="float32", count=1, nodata=-9999.0, compress="deflate")
with rasterio.open(streams_path, "w", **prof) as dst:
    dst.write(stream_arr, 1)

print(f"Stream cells: {int(stream_arr.sum()):,} "
      f"({int(stream_arr.sum()) * DEM_RES_M / 1000:,.0f} km of channel)")

In [ ]:
# ---- The chain: breach -> HAND -> slope ----
breached_path = OUT_DIR / f"{RUN_NAME}_dem_breached_{DEM_RES_M}m.tif"
hand_path     = OUT_DIR / f"{RUN_NAME}_hand_{DEM_RES_M}m.tif"
slope_path    = OUT_DIR / f"{RUN_NAME}_slope_{DEM_RES_M}m.tif"

n = lambda p: p.name   # WBT resolves bare filenames against its working directory

def run_step(name, out_path, fn):
    """Run one WBT step unless its output is already on disk."""
    if out_path.exists():
        print(f"  [cached] {name}")
        return
    t0 = time.time()
    print(f"  [run   ] {name} …", end="", flush=True)
    rc = fn()
    if rc != 0 or not out_path.exists():
        raise RuntimeError(f"WhiteboxTools step '{name}' failed (exit {rc}). "
                           f"Set wbt.verbose = True to see the tool output.")
    print(f" {time.time() - t0:.0f} s")

t_start = time.time()
run_step("BreachDepressionsLeastCost", breached_path, lambda: wbt.breach_depressions_least_cost(
    n(DEM_PATH), n(breached_path), dist=BREACH_DIST_CELLS, fill=True))

run_step("ElevationAboveStream (HAND)", hand_path, lambda: wbt.elevation_above_stream(
    n(breached_path), n(streams_path), n(hand_path)))

run_step("Slope", slope_path, lambda: wbt.slope(
    n(breached_path), n(slope_path), units="degrees"))

HYDRO_SECONDS = time.time() - t_start
print(f"\nHydrology done in {HYDRO_SECONDS:.0f} s")

## 7. Check the flowlines against the terrain

HAND must be ~0 m at every stream cell; if it is not, the flowlines are not sitting in the
DEM thalweg and need snapping. At Red Shirt the median offset was **2.5 m**, so no snapping
was needed there — but §17 of the plan flags this as something to **re-check beyond 13TFJ**,
and this is the cell that does it.

In [ ]:
with rasterio.open(hand_path) as src:
    hand = src.read(1, masked=True)
with rasterio.open(slope_path) as src:
    slope = src.read(1, masked=True)

on_stream = stream_arr == 1
hand_on   = np.ma.filled(hand, np.nan)[on_stream]

print(f"HAND at stream cells : median {np.nanmedian(hand_on):.2f} m, "
      f"90th pct {np.nanpercentile(hand_on, 90):.2f} m")
print(f"HAND nodata          : {100 * np.ma.getmaskarray(hand).mean():.1f}% "
      f"(expect a thin rim at the tile edge)")
print(f"Slope                : median {np.ma.median(slope):.1f} deg")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), constrained_layout=True)
for ax, (arr, title, cmap, vmax) in zip(axes, [
        (np.ma.filled(dem, np.nan),   "DEM (m)",        "terrain", None),
        (np.ma.filled(hand, np.nan),  "HAND (m)",       "viridis", 40),
        (np.ma.filled(slope, np.nan), "Slope (deg)",    "magma",   30)]):
    im = ax.imshow(arr, cmap=cmap, extent=ext, origin="upper", vmax=vmax)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
savefig("dem_hand_slope")
plt.show()

## 8. Valley bottom

A cell is valley bottom if it is **near a reach**, **low above the stream**, and **not
steep** — with all three thresholds taken from that reach's drainage-area class.

In [ ]:
valid = ~np.ma.getmaskarray(hand) & ~np.ma.getmaskarray(slope)
hand_arr, slope_arr = np.ma.filled(hand, 9999.0), np.ma.filled(slope, 90.0)
valley_mask = np.zeros(dem_shape, dtype=np.uint8)

for c in VBET_CLASSES:
    sel = flw[flw.vbet_class == c["label"]]
    if c["label"] not in ANALYSIS_CLASSES:
        excl = sel.geometry.length.sum() / 1000
        print(f"  {c['label']:6s}: {len(sel):5,} reaches ({excl:,.0f} km) excluded from the mask")
        continue
    if len(sel) == 0:
        print(f"  {c['label']:6s}: no reaches")
        continue

    # Rasterizing overlapping buffers to the same value already unions them —
    # no need to union the geometries first.
    near = rasterize(((g, 1) for g in sel.geometry.buffer(c["buffer_m"])),
                     out_shape=dem_shape, transform=dem_transform,
                     fill=0, dtype=np.uint8, all_touched=True)

    cls_mask = (near == 1) & (hand_arr < c["hand_m"]) & (slope_arr < c["slope_deg"]) & valid
    np.maximum(valley_mask, cls_mask, out=valley_mask, casting="unsafe")
    print(f"  {c['label']:6s}: {int(cls_mask.sum()):9,} cells "
          f"({int(cls_mask.sum()) * DEM_RES_M ** 2 / 1e6:7.1f} km2)")

print(f"\nCombined: {int(valley_mask.sum()):,} cells "
      f"({int(valley_mask.sum()) * DEM_RES_M ** 2 / 1e6:.1f} km2)")

In [ ]:
# ---- Clean up and vectorize ----
valley = binary_fill_holes(valley_mask).astype(np.uint8)   # fill enclosed upland pockets

labeled, n_found = label(valley)
min_cells = int((MIN_PATCH_HA * 1e4) / DEM_RES_M ** 2)
counts = np.bincount(labeled.ravel())
counts[0] = 0                                              # label 0 is background
keep = np.flatnonzero(counts >= min_cells)
valley = np.isin(labeled, keep).astype(np.uint8)

polys = [shape(g) for g, v in shapes(valley, mask=valley.astype(bool),
                                     transform=dem_transform) if v == 1]
valley_poly = unary_union(polys).buffer(DEM_RES_M * 2).buffer(-DEM_RES_M * 2)  # smooth edges

# Keep the patches as separate features rather than one dissolved blob. Later steps
# screen and summarise patch by patch, which a single merged polygon cannot support.
parts = list(getattr(valley_poly, "geoms", [valley_poly]))
patches = gpd.GeoDataFrame(geometry=parts, crs=CRS_PROJ)
patches = patches[patches.area > 0].reset_index(drop=True)
patches.insert(0, "patch_id", patches.index + 1)
patches["area_ha"] = patches.area / 1e4
VALLEY_KM2 = patches.area.sum() / 1e6
TILE_KM2   = tile_proj.area.sum() / 1e6

# This fraction is the number that sets the real scaling problem (plan 13) -- it is what
# turns "a 100 km square" into an hours-or-days estimate for the NAIP work in notebook 03.
VALLEY_FRAC = VALLEY_KM2 / TILE_KM2

print(f"Components: {n_found:,} -> {len(keep):,} (kept >= {MIN_PATCH_HA} ha)")
print(f"Patches written: {len(patches):,}  "
      f"(largest {patches.area_ha.max():,.0f} ha, median {patches.area_ha.median():.1f} ha)")
print(f"Valley bottom: {VALLEY_KM2:,.1f} km2 ({100 * VALLEY_FRAC:.1f}% of the square)")

## 9. Save and record the run

In [ ]:
# ---- Binary mask raster, for clipping imagery later ----
# Burn the SAME smoothed polygon that gets written to the GeoPackage, so the raster and
# the vector describe the same area.
MASK_PATH = OUT_DIR / f"{RUN_NAME}_valley_mask_{DEM_RES_M}m.tif"

valley_mask_out = rasterize(((g, 1) for g in patches.geometry), out_shape=dem_shape,
                            transform=dem_transform, fill=0, dtype="uint8")

prof = dem_profile.copy()
prof.update(dtype="uint8", count=1, nodata=255, compress="deflate")
with rasterio.open(MASK_PATH, "w", **prof) as dst:
    dst.write(valley_mask_out, 1)
    dst.write_colormap(1, {0: (245, 245, 245), 1: (46, 125, 50)})

print(f"Mask -> {MASK_PATH.name}  ({valley_mask_out.sum():,} cells)")

fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True)
ax.imshow(np.ma.masked_where(hand > 40, hand), cmap="Greys_r", extent=ext, origin="upper")
gpd.GeoSeries([valley_poly], crs=CRS_PROJ).plot(
    ax=ax, facecolor="#2E7D32", edgecolor="#1B5E20", alpha=0.45, linewidth=0.7)
flw.plot(ax=ax, color="#1565C0", linewidth=0.6)
tile_proj.boundary.plot(ax=ax, color="k", linewidth=1.2, linestyle="--")
ax.set_title(f"{TILE} valley bottom — {VALLEY_KM2:,.1f} km2 ({100*VALLEY_FRAC:.1f}%)")
ax.set_xticks([]); ax.set_yticks([])
savefig("valley_bottom")
plt.show()

In [ ]:
def git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO,
                              capture_output=True, text=True).stdout.strip() or None
    except Exception:
        return None

patches["method"]    = "VBET-simplified (NHD streams)"
patches["run_name"]  = RUN_NAME
patches["dem_res_m"] = DEM_RES_M
patches.to_file(OUTPUT_GPKG, layer="valley_bottom", driver="GPKG")
flw.to_file(OUTPUT_GPKG, layer="flowlines_classed", driver="GPKG")
tile_proj.to_file(OUTPUT_GPKG, layer="mgrs_tile", driver="GPKG")
print(f"Valley bottom -> {OUTPUT_GPKG.relative_to(REPO)}")

manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "01_VBET_ValleyBottom.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit":  git_commit(),
    "inputs": {
        "dem_tiles":       TILE_URLS,
        "dem_product":     DEM_PRODUCT,
        "flowline_source": FLOWLINE_SOURCE,
        "n_reaches":       int(len(flw)),
    },
    "parameters": {
        "tile":              TILE,
        "tile_buffer_m":     TILE_BUFFER_M,
        "crs":               CRS_PROJ,
        "dem_res_m":         DEM_RES_M,
        "breach_dist_cells": BREACH_DIST_CELLS,
        "vbet_classes":      VBET_CLASSES,
        "min_patch_ha":      MIN_PATCH_HA,
        "analysis_classes":  ANALYSIS_CLASSES,
    },
    "environment": {
        "python":        sys.version.split()[0],
        "whiteboxtools": WBT_VERSION,
        "geopandas":     gpd.__version__,
        "rasterio":      rasterio.__version__,
    },
    "results": {
        "valley_bottom_km2":   round(VALLEY_KM2, 3),
        "valley_frac_of_tile": round(VALLEY_FRAC, 4),
        "n_patches":           int(len(patches)),
        "tile_km2":            round(TILE_KM2, 1),
        "hydrology_seconds":   round(HYDRO_SECONDS, 1),
        "hand_median_on_stream_m": round(float(np.nanmedian(hand_on)), 3),
    },
    "outputs": [OUTPUT_GPKG.name, MASK_PATH.name, hand_path.name, slope_path.name],
}

manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Manifest      -> {manifest_path.relative_to(REPO)}")
print(json.dumps(manifest["results"], indent=2))

## What comes next

Phase 1b gate: does the valley bottom follow the floodplain rather than climbing terraces?
Check it against NAIP in notebook `02` before anything downstream trusts it.

Then set `VBET_RUN = "vbet_13TFJ"` in notebook `03` — that is the only edit needed to move
the label factory off the smoke test and onto the pilot tile.